In [1]:
# 1. Lade das Backend (jetzt ZWEI Dateien!)
%run deduplication.ipynb
from author_cleaning import remove_ghost_papers

def build_cv_pipeline(author_id, end_year=2026):
    print(f"Starte Pipeline für ID: {author_id}...")
    
    # 2. Rohe Daten laden
    raw_data = fetch_openalex_data(author_id)
    df_raw = extract_flat_data(raw_data, author_id)
    if df_raw.empty: return pd.DataFrame()
    
    # 3. PIPELINE SCHRITT A: Autor-Bereinigung (Geister entfernen)
    df_author_clean = remove_ghost_papers(df_raw)
    
    # 4. PIPELINE SCHRITT B: Paper-Deduplizierung (Working Papers entfernen)
    df_clean = deduplicate_papers(df_author_clean)
    
    # 5. Pivot-Tabelle bauen
    df_clean['Inst_Nummer'] = df_clean.groupby('Jahr').cumcount() + 1
    df_clean['Inst_Nummer'] = df_clean['Inst_Nummer'].astype(int)

    df_pivot = df_clean.pivot(index='Jahr', columns='Inst_Nummer', values=['Uni', 'Titel', 'Link'])

    neue_spalten = []
    for col in df_pivot.columns:
        if col[0] == 'Uni': neue_spalten.append(f'Institution_{int(col[1])}')
        elif col[0] == 'Titel': neue_spalten.append(f'Beweis_Paper_{int(col[1])}')
        elif col[0] == 'Link': neue_spalten.append(f'Beweis_Link_{int(col[1])}')
            
    df_pivot.columns = neue_spalten
    df_pivot = df_pivot.reset_index()

    max_inst = int(df_clean['Inst_Nummer'].max()) 
    gewuenschte_reihenfolge = ['Jahr']
    for i in range(1, max_inst + 1):
        if f'Institution_{i}' in df_pivot.columns: gewuenschte_reihenfolge.append(f'Institution_{i}')
        if f'Beweis_Paper_{i}' in df_pivot.columns: gewuenschte_reihenfolge.append(f'Beweis_Paper_{i}')
        if f'Beweis_Link_{i}' in df_pivot.columns: gewuenschte_reihenfolge.append(f'Beweis_Link_{i}')
            
    df_pivot = df_pivot[gewuenschte_reihenfolge]
    start_year = int(df_pivot['Jahr'].min())
    zeitstrahl = pd.DataFrame({'Jahr': range(start_year, end_year + 1)})
    
    return pd.merge(zeitstrahl, df_pivot, on='Jahr', how='left').fillna("-")

# --- AUSFÜHRUNG ---
test_author_id = "A5058372080" # Klaus Schmidt
cv_tabelle_final = build_cv_pipeline(test_author_id)
cv_tabelle_final

Lade und verarbeite Daten...
Starte Pipeline für ID: A5058372080...

--- 👻 GHOST AUTHOR SCAN ---
🌍 Makro-Domäne gesichert: 'Social Sciences'
🔬 Thematischer Kern (Top 2 Fields): ['Social Sciences', 'Economics, Econometrics and Finance']
🗑️ Blockiert (Biochemistry, Genetics and Molecular Biology | [RAW] Department of ...): 'Genome-wide association analyses of risk...'
🗑️ Blockiert (Engineering | Illinois State Unive...): 'Fuel Cell Electric Vehicles (FCEV): Poli...'
🗑️ Blockiert (Business, Management and Accounting | [RAW] University of ...): 'Convertible Securities and Venture Capit...'
🗑️ Blockiert (Medicine | [RAW] Abteilung für ...): 'Vergleichende Endoprothetik des rheumati...'
🗑️ Blockiert (Medicine | Ruhr University Boch...): 'The correlation between magnetic resonan...'
🗑️ Blockiert (Decision Sciences | [RAW] Department of ...): 'Procurement with Unforeseen Contingencie...'
🗑️ Blockiert (Engineering | Illinois State Unive...): 'Designing a new course using Backward de...'
🗑️ Bloc

,Jahr,Institution_1,Beweis_Paper_1,Beweis_Link_1,Institution_2,Beweis_Paper_2,Beweis_Link_2,Institution_3,Beweis_Paper_3,Beweis_Link_3,...,Beweis_Link_4,Institution_5,Beweis_Paper_5,Beweis_Link_5,Institution_6,Beweis_Paper_6,Beweis_Link_6,Institution_7,Beweis_Paper_7,Beweis_Link_7
0,1976,Ludwig-Maximilians-Universität München,Zur kaledonischen Orogenese in den Ostalpen,https://api.openalex.org/W2062891553,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
1,1977,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
2,1978,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
3,1979,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
4,1980,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
5,1981,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
6,1982,Ludwig-Maximilians-Universität München,Zur Trias des Iran,https://api.openalex.org/W2047503001,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
7,1983,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
8,1984,Ludwig-Maximilians-Universität München,"Zur Geologie des Thurntaler Quarzphyllits und des Altkristallins südlich des Tauernfensters (Ostalpen, Südtirol)",https://api.openalex.org/W1984253050,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
9,1985,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
